# 张量、形状与设备

## 学习目标

能够解释 shape、stride、dtype、广播和设备移动，并在运行前预测输出形状。


## 概念模型与执行路径

张量由数据、形状、数据类型、布局和设备共同定义。神经网络的多数错误不是公式错误，而是这些属性不匹配。广播从末尾维度对齐；设备移动会创建新的张量，模型参数和输入必须位于同一设备。


### 实验 1


In [ ]:
import torch
from common.runtime import choose_device, seed_everything

seed_everything(42)
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("shape:", x.shape, "stride:", x.stride(), "dtype:", x.dtype)
print(x)


### 实验 2


In [ ]:
bias = torch.tensor([10.0, 20.0, 30.0, 40.0])
result = x + bias
print("broadcast result shape:", result.shape)
print(result)


### 实验 3


In [ ]:
view = x[:, :2]
clone = view.clone()
view[0, 0] = -99
print("view shares storage:", x[0, 0].item())
print("clone owns copied data:", clone[0, 0].item())


### 实验 4


In [ ]:
device = choose_device("auto")
x_device = x.to(device)
print("selected device:", device)
print("tensor device:", x_device.device)
print("back on CPU as NumPy:", x_device.cpu().numpy().shape)


## 底层机制

`reshape` 尽量返回视图，但布局不兼容时可能复制；`view` 要求兼容 stride。`.to(device)` 不会原地修改原变量。GPU 计算前后频繁搬运数据会抵消加速收益。


## 检查点

在不运行代码的情况下，预测 `(2, 3, 4) + (4,)` 的输出形状。答案应为 `(2, 3, 4)`。


## 试一试

把 `bias` 改成 shape 为 `(3, 1)` 的张量并解释为何仍能广播。再尝试 `(2,)`，记录 PyTorch 给出的维度错误。


## 常见错误与调试

混用 float32/float64、把输入留在 CPU 而模型在 CUDA、误以为切片一定复制、使用 `squeeze()` 意外删除 batch 维。
